In [ ]:
import pickle
import numpy as np

with open(r'C:\Users\Ahad Imran\Desktop\CS7180-Final\data\pie_database.pkl', 'rb') as f:
    db = pickle.load(f)

print(f'Type: {type(db)}')
if isinstance(db, dict):
    print(f'Top-level keys ({len(db)}): {list(db.keys())}')
elif isinstance(db, list):
    print(f'Length: {len(db)}')
    print(f'First element type: {type(db[0])}')

In [ ]:
# Recursively print the structure of nested dicts/lists up to a given depth
def print_structure(obj, name='root', depth=0, max_depth=4):
    indent = '  ' * depth
    if depth > max_depth:
        print(f'{indent}{name}: ...')
        return
    if isinstance(obj, dict):
        print(f'{indent}{name}: dict ({len(obj)} keys)')
        for k in list(obj.keys())[:8]:  # show first 8 keys
            print_structure(obj[k], name=str(k), depth=depth+1, max_depth=max_depth)
        if len(obj) > 8:
            print(f'{indent}  ... and {len(obj) - 8} more keys')
    elif isinstance(obj, list):
        print(f'{indent}{name}: list (len={len(obj)})')
        if len(obj) > 0:
            print_structure(obj[0], name='[0]', depth=depth+1, max_depth=max_depth)
    elif isinstance(obj, np.ndarray):
        print(f'{indent}{name}: np.ndarray shape={obj.shape} dtype={obj.dtype}')
    else:
        print(f'{indent}{name}: {type(obj).__name__} = {repr(obj)[:80]}')

print_structure(db)

In [ ]:
# If the database is a dict of sets/videos, explore the first entry fully
if isinstance(db, dict):
    first_key = list(db.keys())[0]
    print(f'Drilling into first key: "{first_key}"')
    print_structure(db[first_key], name=first_key, max_depth=5)

In [ ]:
# Count total pedestrian samples and check label distributions
# Adjust key path below based on structure revealed above

all_cross_labels = []
all_ego_speeds   = []
total_peds = 0

def collect_labels(obj):
    """Walk the nested structure looking for 'cross' and 'obd_speed' arrays."""
    global total_peds
    if isinstance(obj, dict):
        if 'cross' in obj:
            labels = np.array(obj['cross']).flatten()
            all_cross_labels.extend(labels.tolist())
            total_peds += 1
        if 'obd_speed' in obj:
            speeds = np.array(obj['obd_speed']).flatten()
            all_ego_speeds.extend(speeds.tolist())
        for v in obj.values():
            collect_labels(v)
    elif isinstance(obj, list):
        for item in obj:
            collect_labels(item)

collect_labels(db)

print(f'Total pedestrian tracks found: {total_peds}')

if all_cross_labels:
    labels = np.array(all_cross_labels)
    unique, counts = np.unique(labels, return_counts=True)
    print(f'\nCross label distribution:')
    for u, c in zip(unique, counts):
        print(f'  {int(u)}: {c} ({100*c/len(labels):.1f}%)')

if all_ego_speeds:
    speeds = np.array(all_ego_speeds)
    print(f'\nEgo speed stats:')
    print(f'  min={speeds.min():.2f}  max={speeds.max():.2f}  mean={speeds.mean():.2f}  std={speeds.std():.2f}')

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

if all_cross_labels:
    labels = np.array(all_cross_labels)
    unique, counts = np.unique(labels, return_counts=True)
    axes[0].bar([str(int(u)) for u in unique], counts)
    axes[0].set_title('Cross label distribution')
    axes[0].set_xlabel('Label')
    axes[0].set_ylabel('Count')
else:
    axes[0].text(0.5, 0.5, 'No cross labels found', ha='center', va='center')
    axes[0].set_title('Cross label distribution')

if all_ego_speeds:
    speeds = np.array(all_ego_speeds)
    axes[1].hist(speeds, bins=50, edgecolor='black')
    axes[1].set_title('Ego vehicle speed distribution')
    axes[1].set_xlabel('Speed')
    axes[1].set_ylabel('Frequency')
else:
    axes[1].text(0.5, 0.5, 'No speed data found', ha='center', va='center')
    axes[1].set_title('Ego vehicle speed distribution')

plt.tight_layout()
plt.show()

In [ ]:
# Print all unique top-level keys found anywhere in the database
all_keys = set()

def collect_keys(obj, depth=0, max_depth=6):
    if depth > max_depth:
        return
    if isinstance(obj, dict):
        all_keys.update(obj.keys())
        for v in obj.values():
            collect_keys(v, depth+1, max_depth)
    elif isinstance(obj, list):
        for item in obj[:3]:  # sample first 3 to avoid blowing up on huge lists
            collect_keys(item, depth+1, max_depth)

collect_keys(db)
print(f'All unique keys found in database ({len(all_keys)}):')
for k in sorted(all_keys, key=str):
    print(f'  {k}')